# Bank Marketing — Data Preprocessing

## 1. Introduction

### 1.1 Objective & Scope

This notebook implements a reproducible baseline preprocessing
configuration for the Bank Marketing project.

The objectives are to:

- Reproduce the chronological split established in Notebook 01.
- Define modeling features and preprocessing groups.
- Establish a simple numerical and categorical preprocessing strategy.
- Fit preprocessing on Train and transform Validation.
- Validate the transformed structure and unseen-category handling.

This notebook implements a basic representation suitable for
a linear-model baseline. It does not establish that the
representation is optimal.

Baseline model definition, training, and evaluation are handled
in Notebook 03.

Comparisons of alternative feature representations, preprocessing
strategies, and models are handled in Notebook 04.

### 1.2 Preprocessing Requirements

The configuration follows the established project decisions:

- Prediction time is immediately before a planned marketing contact.
- Exclude `duration` from modeling features.
- Retain the remaining 19 candidate predictors.
- Preserve the chronological 70/20/10 split.
- Fit learned transformations on Train only.
- Transform Validation without refitting.
- Keep Test frozen for final evaluation.
- Preserve `"unknown"` as an explicit category.
- Handle unseen categorical values safely.
- Retain raw `pdays` for the initial representation.
- Avoid automatic clipping, category merging, or feature removal.

This notebook runs independently of the variables and fitted
objects created in Notebook 01.

## 2. Load Data & Splits

### 2.1 Load Dataset

Reload the original CSV without changing row order or feature values.

The path setup supports execution from either the project root
or its `notebooks/` directory.

The raw dataframe retains `duration`; modeling inputs exclude it.

In [84]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import issparse
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [89]:
PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bank-additional"
    / "bank-additional-full.csv"
)

RAW_DATA_PATH

PosixPath('/home/moonk/projects/bank-marketing-ml/data/raw/bank-additional/bank-additional-full.csv')

In [90]:
df = pd.read_csv(RAW_DATA_PATH, sep=";")
df.shape

(41188, 21)

In [8]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### 2.2 Reproduce Established Splits

Preserve the original chronological row order:

- Train: first 70%, containing 28,831 observations.
- Validation: next 20%, containing 8,238 observations.
- Test: final 10%, containing 4,119 observations.

No shuffling, stratification, or sorting is applied.

Test is partitioned only to reproduce the established boundaries.
It is not explored or transformed in this notebook.

In [91]:
n_rows = len(df)

train_end = int(n_rows * 0.70)
valid_end = int(n_rows * 0.90)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()
test_df = df.iloc[valid_end:].copy()

### 2.3 Define Features & Target

Separate candidate predictors from the target.

The modeling inputs exclude:

- `y`: the target.
- `duration`: unavailable before the planned contact.

Encode the development targets as:

- `no` → 0
- `yes` → 1

Target labels are prepared for consistency with subsequent
modeling notebooks. They are not used to fit this preprocessor.

Test features and labels will be prepared in Notebook 05.

In [92]:
target = "y"
excluded_features = ["duration"]

modeling_features = [
    col
    for col in df.columns
    if col not in [target, *excluded_features]
]

X_train = train_df[modeling_features].copy()
X_valid = valid_df[modeling_features].copy()

target_mapping = {"no": 0, "yes": 1}

y_train = train_df[target].map(target_mapping)
y_valid = valid_df[target].map(target_mapping)

### 2.4 Split & Input Validation

Verify the established sizes and complete preservation of row order.

Also check that:

- Train and Validation use the same predictor columns.
- The target and `duration` are excluded.
- Binary target encoding succeeded.
- Feature and target indices remain aligned.

In [93]:
assert (
    len(train_df),
    len(valid_df),
    len(test_df),
) == (28_831, 8_238, 4_119)

combined_index = pd.concat(
    [train_df, valid_df, test_df]
).index

assert combined_index.equals(df.index)

assert len(modeling_features) == 19
assert X_train.columns.equals(X_valid.columns)
assert target not in X_train.columns
assert "duration" not in X_train.columns

assert y_train.notna().all(), "Unexpected or missing Train target labels."
assert y_valid.notna().all(), "Unexpected or missing Validation target labels."

assert X_train.index.equals(y_train.index)
assert X_valid.index.equals(y_valid.index)

split_summary = pd.DataFrame(
    {
        "dataset": ["Train", "Validation", "Test"],
        "rows": [
            len(train_df),
            len(valid_df),
            len(test_df),
        ],
    }
)

display(split_summary)

print(f"Modeling predictors: {len(modeling_features)}")
print("Split and input validation passed.")

,dataset,rows
0,Train,28831
1,Validation,8238
2,Test,4119


Modeling predictors: 19
Split and input validation passed.


## 3. Feature Definition

### 3.1 Define Feature Groups

Assign the 19 candidate predictors to explicit groups:

- Eight numerical features.
- Ten categorical features.
- One special feature: `pdays`.

These groups reflect preprocessing requirements rather than
relying solely on pandas data types.

`pdays` is documented separately because `999` is a sentinel
value. For the initial configuration, it will still be processed
through the numerical branch.

In [94]:
numeric_features = [
    "age",
    "campaign",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

special_features = [
    "pdays",
]

### 3.2 Validate Feature Groups

Every candidate predictor must belong to exactly one group.

Check for duplicated assignments, omitted predictors,
and accidental inclusion of excluded columns.

In [95]:
all_features = (
    numeric_features
    + categorical_features
    + special_features
)

assert len(all_features) == len(set(all_features))
assert set(all_features) == set(modeling_features)
assert target not in all_features
assert "duration" not in all_features

feature_group_summary = pd.DataFrame(
    {
        "group": ["Numerical", "Categorical", "Special"],
        "n_features": [
            len(numeric_features),
            len(categorical_features),
            len(special_features),
        ],
    }
)

feature_group_summary

,group,n_features
0,Numerical,8
1,Categorical,10
2,Special,1


## 4. Preprocessing Strategy

### 4.1 Missing & Unknown Values

Notebook 01 identified no pandas-recognized missing values in Train.

Several categorical predictors contain the literal value
`"unknown"`. These values remain explicit categories.

For the current baseline:

- Do not add an imputer.
- Do not replace `"unknown"` with a missing value.
- Do not remove observations containing `"unknown"`.

Check that both development inputs satisfy the current
no-missing-values assumption.

If actual missing values appear in future data, missing-value
handling will require a separate decision.

In [96]:
assert not X_train.isna().any().any(), (
    "Missing values found in Train features."
)

assert not X_valid.isna().any().any(), (
    "Missing values found in Validation features."
)

print("No pandas-recognized missing values in development features.")

No pandas-recognized missing values in development features.


### 4.2 Categorical Encoding

Use:

`OneHotEncoder(handle_unknown="ignore")`

One-hot encoding provides a simple representation without
imposing arbitrary numerical distances between categories.

The configuration:

- Learns categories from Train only.
- Retains all observed category levels.
- Uses the default `drop=None`.
- Does not merge rare categories.
- Allows sparse output.

Distinguish the following cases:

| Case | Encoding Behavior |
|---|---|
| Literal `"unknown"` observed in Train | Receives its own one-hot column. |
| A category not observed in Train | Produces zeros across that predictor's one-hot columns. |

`handle_unknown="ignore"` prevents transformation errors for
unseen categories. It does not learn a separate effect for
each new category.

### 4.3 Numerical Transformation

Use:

`StandardScaler()`

The scaler learns each numerical predictor's mean and scale
from Train and applies the same transformation to Validation.

Scaling is useful for a linear-model baseline because
numerical predictors have different units and magnitudes,
and regularization is affected by feature scale.

Standardization changes location and scale. It does not
make a distribution normal, remove skewness, or eliminate outliers.

The initial configuration does not apply:

- Log transformations.
- Clipping or winsorization.
- Outlier removal.
- Correlation-based feature removal.

Tree-based models may use a different numerical preprocessing
strategy because scaling is generally unnecessary for them.

The baseline estimator will be defined in Notebook 03.

### 4.4 Special Feature: `pdays`

Approximately 99.82% of Train has `pdays = 999`.

The sentinel is not an ordinary elapsed-day observation.
Its relationship with other campaign-history predictors
was documented in Notebook 01.

For the initial representation:

- Retain the original numerical values.
- Include `pdays` in the StandardScaler branch.
- Do not infer replacement values from `previous` or `poutcome`.
- Do not create additional sentinel or recency predictors here.

This representation preserves the existing baseline configuration.
Standardization does not resolve the sentinel's meaning.

Its concentration also means that non-sentinel observations
can have large absolute standardized values.

Alternative representations will be compared in Notebook 04.

### 4.5 Baseline Preprocessing Configuration

| Component | Initial Configuration |
|---|---|
| Eight numerical features | StandardScaler |
| `pdays` | Original numerical values with StandardScaler |
| Ten categorical features | OneHotEncoder(handle_unknown="ignore") |
| Unspecified columns | Drop |

The configuration uses all 19 candidate predictors.

No additional feature engineering, category merging,
or outlier treatment is applied.

This is a basic configuration to carry into baseline modeling,
not a performance-based selection among preprocessing alternatives.

In [97]:
raw_numeric_features = numeric_features + special_features

assert len(raw_numeric_features) == 9
assert set(raw_numeric_features).isdisjoint(categorical_features)

assert set(
    raw_numeric_features + categorical_features
) == set(modeling_features)

## 5. Preprocessing Pipeline

### 5.1 Build Baseline Preprocessor

Combine the numerical and categorical branches using
`ColumnTransformer`.

Only explicitly assigned predictors are transformed.

The object defined below is initially unfitted.
It has not yet learned numerical statistics or category levels.

In [98]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            raw_numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ],
    remainder="drop",
)

### 5.2 Fit on Train & Transform Validation

Fit the preprocessor once on Train:

- StandardScaler learns numerical means and scales.
- OneHotEncoder learns observed category levels.

Transform Validation with the fitted object.

Do not call `fit` or `fit_transform` on Validation.

The target is not required because these transformations
do not use target labels.

In [99]:
X_train_prepared = preprocessor.fit_transform(X_train)

X_valid_prepared = preprocessor.transform(X_valid)

### 5.3 Validate Transformed Structure

Check that transformation:

- Preserves observation counts.
- Produces the same feature width for Train and Validation.
- Provides unique transformed feature names.
- Excludes the target and `duration`.
- Produces no NaN or infinite values.

The final output may be dense or sparse. ColumnTransformer
chooses the combined format based on its output density.

Validation checks therefore support both formats without
converting the full output to a dense matrix.

In [100]:
feature_names = preprocessor.get_feature_names_out()

assert pd.Index(feature_names).is_unique
assert list(preprocessor.feature_names_in_) == modeling_features
assert target not in preprocessor.feature_names_in_
assert "duration" not in preprocessor.feature_names_in_

for label, prepared, original in [
    ("Train", X_train_prepared, X_train),
    ("Validation", X_valid_prepared, X_valid),
]:
    assert prepared.shape == (
        len(original),
        len(feature_names),
    )

    stored_values = prepared.data if issparse(prepared) else prepared

    assert np.isfinite(stored_values).all(), (
        f"Non-finite transformed values found in {label}."
    )

    output_format = "sparse" if issparse(prepared) else "dense"

    print(
        f"{label}: shape={prepared.shape}, "
        f"format={output_format}, dtype={prepared.dtype}"
    )

print("Transformed structure validation passed.")

Train: shape=(28831, 61), format=dense, dtype=float64
Validation: shape=(8238, 61), format=dense, dtype=float64
Transformed structure validation passed.


Inspect the fitted branches to explain the transformed width.

The numerical branch preserves one column per numerical predictor.

The categorical branch creates one column per training category
because no reference category or rare-category grouping is used.

In [101]:
fitted_scaler = preprocessor.named_transformers_["num"]
fitted_encoder = preprocessor.named_transformers_["cat"]

assert np.all(
    fitted_scaler.n_samples_seen_ == len(X_train)
)

n_encoded_categories = sum(
    len(categories)
    for categories in fitted_encoder.categories_
)

assert len(feature_names) == (
    len(raw_numeric_features) + n_encoded_categories
)

transformation_summary = pd.DataFrame(
    {
        "branch": ["Numerical", "Categorical"],
        "input_features": [
            len(raw_numeric_features),
            len(categorical_features),
        ],
        "output_features": [
            len(raw_numeric_features),
            n_encoded_categories,
        ],
    }
)

display(transformation_summary)

print(f"Total transformed features: {len(feature_names)}")

,branch,input_features,output_features
0,Numerical,9,9
1,Categorical,10,52


Total transformed features: 61


### 5.4 Validate Unseen-Category Handling

Inspect which Validation category values were not learned from Train.

This is an encoder compatibility check. Validation categories
are not added to the fitted encoder, and the check does not
use target labels or predictive performance.

In [102]:
category_check_rows = []

for feature, learned_categories in zip(
    categorical_features,
    fitted_encoder.categories_,
):
    unseen_values = sorted(
        set(X_valid[feature]) - set(learned_categories)
    )

    category_check_rows.append(
        {
            "feature": feature,
            "train_categories": len(learned_categories),
            "unseen_valid_categories": unseen_values,
            "unseen_valid_rows": int(
                X_valid[feature].isin(unseen_values).sum()
            ),
        }
    )

category_checks = (
    pd.DataFrame(category_check_rows)
    .set_index("feature")
)

category_checks

,train_categories,unseen_valid_categories,unseen_valid_rows
feature,,,
job,12,[],0
marital,4,[],0
education,8,[],0
default,3,[],0
housing,3,[],0
loan,3,[],0
contact,2,[],0
month,9,[],0
day_of_week,5,[],0


Also verify unseen-category behavior with a synthetic input.

Copy one training observation and replace its job category
with a value absent from the fitted encoder.

The expected behavior is:

- Transformation succeeds.
- The transformed width is unchanged.
- All job one-hot columns are zero.

The original data and fitted preprocessing parameters
are not modified.

In [103]:
unseen_job = "__unseen_job_for_validation__"

job_feature_index = categorical_features.index("job")

assert unseen_job not in (
    fitted_encoder.categories_[job_feature_index]
)

unseen_probe = X_train.iloc[[0]].copy()
unseen_probe["job"] = unseen_job

probe_prepared = preprocessor.transform(unseen_probe)

job_columns = [
    i
    for i, name in enumerate(feature_names)
    if name.startswith("cat__job_")
]

assert job_columns
assert probe_prepared.shape == (1, len(feature_names))
assert probe_prepared[:, job_columns].sum() == 0

print("Unseen-category handling validation passed.")

Unseen-category handling validation passed.


## 6. Preprocessing Summary

### 6.1 Baseline Preprocessing Configuration

The implemented configuration is:

| Component | Configuration |
|---|---|
| Numerical branch | StandardScaler on eight numerical predictors and raw `pdays` |
| Categorical branch | OneHotEncoder(handle_unknown="ignore") on ten predictors |
| Other columns | Drop |

The configuration retains all 19 candidate predictors
and excludes `duration`.

Initial decisions:

- Preserve `"unknown"` as an explicit category.
- Retain all observed categorical levels.
- Handle unseen values without transformation errors.
- Preserve raw `pdays`.
- Retain rare categories, extreme values, and correlated predictors.

### 6.2 Preprocessing Validation

The validation checks cover:

- Reproduction of the chronological split.
- Complete and non-overlapping feature-group assignments.
- Successful binary target encoding for development data.
- Fitting learned preprocessing on Train only.
- Consistent transformed dimensions and feature names.
- Absence of NaN and infinite transformed values.
- Compatibility with dense and sparse output.
- Unseen-category handling using a synthetic input.

With the established training categories, the configuration
produces 61 transformed features:

- Nine numerical columns.
- 52 one-hot categorical columns.

The transformed width is derived from the fitted training
categories rather than hard-coded into the validation checks.

Test is not transformed or evaluated.

### 6.3 Design Decisions & Limitations

**Design decisions**

This notebook establishes a simple, reproducible representation
before predictive performance comparisons.

No estimator is fitted, and no preprocessing option is selected
using validation metrics.

**Known limitations**

- StandardScaler is intended for the linear-model baseline
  representation and may be unnecessary for tree-based models.
- Standardization does not remove skewness or make scaling
  robust to extreme values.
- Raw `pdays` retains the numerical distances introduced
  by its sentinel value.
- An unseen category receives no category-specific learned effect.
- Actual missing values are not handled by a dedicated imputer.
- Temporal distribution shift remains present after preprocessing.

These limitations define possible experiments; they do not
automatically justify changing the initial configuration.

### 6.4 Next Steps

`03-baseline-modeling.ipynb` will:

1. Reload the raw data and reproduce the established split.
2. Reconstruct a fresh preprocessor with the same configuration.
3. Define the baseline estimator.
4. Combine preprocessing and the estimator in a full sklearn Pipeline.
5. Fit the Pipeline on raw training predictors and binary labels.
6. Generate positive-class probabilities for Validation.
7. Evaluate AP, ROC-AUC, Precision@K, Recall@K, and Lift@K.
8. Document the baseline configuration and results.

The prepared matrices in this notebook demonstrate preprocessing
behavior. Notebook 03 will use raw feature dataframes as inputs
to the full Pipeline, which handles preprocessing internally.

Notebook 03 must not depend on this notebook's in-memory variables
or fitted objects.

`04-model-experiments.ipynb` will compare alternative feature
representations, preprocessing choices, models, and hyperparameters.

Test remains frozen for final evaluation in Notebook 05.